# 上传视频，生成增量语义 Gaussian 地图
选择 NVIDIA GPU（L4），依次运行。完整源码已嵌入，无须另行上传代码。

自动抽帧、估计相机轨迹、去畸变，随后增量训练并生成物品与细区域分割、PLY 和交互回放。
相机估计先读取完整视频，因此本版本是离线 SfM + 增量 Gaussian 建图，不是实时 SLAM。


In [ ]:
#@title 1. 解包完整源码
from pathlib import Path
import base64,hashlib,zipfile,io,subprocess,sys
ROOT=Path('/content/video-gaussian-pipeline');ROOT.mkdir(exist_ok=True)
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='284bb6d126e1a9bd5f9c8d0053fb700c569c44f764b3a9f3190179d03917a3c3'
zipfile.ZipFile(io.BytesIO(blob)).extractall(ROOT)
print('Source ready',ROOT)


In [ ]:
#@title 2. 安装依赖及 SAM2 权重
subprocess.run([sys.executable,str(ROOT/'install_colab.py')],check=True)
subprocess.run([sys.executable,'-m','videogs','doctor'],cwd=ROOT,check=True)


In [ ]:
#@title 3. 上传自己的视频，或使用真实录像样例
USE_DEMO = False #@param {type:"boolean"}
if USE_DEMO:
    VIDEO=ROOT/'videos/tum-desk.mp4'
    subprocess.run([sys.executable,'-m','videogs','demo','--output',str(VIDEO),'--seconds','24'],cwd=ROOT,check=True)
else:
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('每次请选择一个视频文件')
    name,content=next(iter(uploaded.items()))
    VIDEO=ROOT/'videos'/Path(name).name
    VIDEO.parent.mkdir(exist_ok=True);VIDEO.write_bytes(content)
    del uploaded,content
print('Input video',VIDEO)


In [ ]:
#@title 4. 运行完整链路（后台日志支持检查）
FPS = 3.0 #@param {type:"number"}
MAX_FRAMES = 180 #@param {type:"integer"}
MAX_GAUSSIANS = 500000 #@param {type:"integer"}
STEPS_PER_STAGE = 2400 #@param {type:"integer"}
IMPORTANT_OBJECTS = "monitor,keyboard,mouse,cup,bottle,book,laptop" #@param {type:"string"}
BACKGROUND_PRUNE_FRACTION = 0.03 #@param {type:"number"}
STABILITY_MODE = "mature" #@param ["mature", "schedule", "legacy"]
BATCH_WARMUP = 200 #@param {type:"integer"}
BATCH_SETTLE = 800 #@param {type:"integer"}
RUN_NAME = "my-video-01" #@param {type:"string"}
if Path(RUN_NAME).name!=RUN_NAME or RUN_NAME in ('','.','..'):raise ValueError('RUN_NAME 应为单个目录名')
if 'job' in globals() and job.poll() is None:raise RuntimeError('已有任务正在运行，请先查看进度，避免重复启动')
OUT=ROOT/'results'/RUN_NAME;OUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,'-m','videogs','run','--video',str(VIDEO),'--output',str(OUT),
     '--fps',str(FPS),'--max-frames',str(MAX_FRAMES),'--cap',str(MAX_GAUSSIANS),'--steps',str(STEPS_PER_STAGE),'--important',IMPORTANT_OBJECTS,
     '--background-prune-fraction',str(BACKGROUND_PRUNE_FRACTION),'--stability-mode',STABILITY_MODE,
     '--batch-warmup',str(BATCH_WARMUP),'--batch-settle',str(BATCH_SETTLE)]
job=subprocess.Popen(cmd,cwd=ROOT,stdout=open(OUT/'console.log','w'),stderr=subprocess.STDOUT,start_new_session=True)
print('Started PID',job.pid,'Results',OUT)


In [ ]:
#@title 5. 查看进度（可以重复运行）
import json
print('Process',job.poll())
p=OUT/'pipeline.json'
if p.exists():
    state=json.loads(p.read_text());print(state['state'],state.get('current_stage'),state.get('error',''))
    print(json.dumps(state['stages'],indent=2))
for name in ['console.log','logs/teachers.log','logs/semantics.log','logs/training.log']:
    p=OUT/name
    if p.exists():print(name,p.read_text(errors='replace')[-1800:])


In [ ]:
#@title 6. 打开交互回放
import functools,http.server,threading
import sys
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from videogs.server import PlaybackHandler
from google.colab import output
if not (OUT/'summary.json').exists():raise RuntimeError('请等待运行成功完成；用上方单元格检查失败日志')
if 'viewer_server' in globals():viewer_server.shutdown();viewer_server.server_close()
viewer_server=http.server.ThreadingHTTPServer(('127.0.0.1',8770),functools.partial(PlaybackHandler,directory=str(OUT)))
threading.Thread(target=viewer_server.serve_forever,daemon=True).start()
output.serve_kernel_port_as_iframe(8770,path='/viewer.html',height=1050)


In [ ]:
#@title 7. 下载完整结果
from google.colab import files
import shutil
if json.loads((OUT/'pipeline.json').read_text())['state']!='complete':raise RuntimeError('运行尚未完成')
archive=shutil.make_archive(str(ROOT/(RUN_NAME+'-results')),'zip',OUT)
files.download(archive)


输出包括完整 Gaussian PLY、NPZ、最终模型检查点、相机轨迹、分割层级、原始指标与日志。Colab 临时磁盘会回收，请及时下载。

物品和细区域是 SAM/CLIP 的预测；PSNR 留出图像仍参与 SfM。3D 交互窗口加载每阶段完整各向异性高斯，默认真实拍摄视角，可环绕、缩放和聚焦物品；颜色使用 SH 的 DC 分量。下方图片使用完整 SH 的 CUDA 渲染。

重建期间学习每高斯 16 维压缩 CLIP 特征，融合物品/细区域标签。每批图像根据重要物品与边界调整增密；仅裁剪已确认背景的低贡献尾部，并保护空间覆盖。高斯上限 500,000，裁剪比例是上限而非必须达到的目标。单目地图尺度不能直接解释为米。
